# Ablation Study: DenseNet201 + Augmentation
## With Augmentation, No FPN, No Class Weighting, No GCA

This notebook contains:
- DenseNet201 backbone
- **Data augmentation enabled**
- Simple classification head
- No FPN, No class weighting, No GCA

In [ ]:
import os
import sys
import random
import warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from collections import Counter
import json
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

# PyTorch imports
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Torchvision imports
from torchvision import transforms
from torchvision.models import densenet201, DenseNet201_Weights

# Albumentations for augmentation
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Scikit-learn for metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score
)

# Suppress warnings
warnings.filterwarnings('ignore')

In [ ]:
def set_seed(seed: int = 42):
    """Set random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(42)

# Device configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"📊 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# CONFIGURATION PARAMETERS
class Config:
    """Configuration with augmentation enabled."""
    
    # Paths
    DATA_ROOT = Path("../dataset/Raw_dataset")
    OUTPUT_DIR = Path("outputs_augmentation")
    CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
    LOG_DIR = OUTPUT_DIR / "logs"
    
    # Dataset parameters
    CLASSES = ['Healthy', 'Anthracnose']
    NUM_CLASSES = 2
    PLANTS = ['Guava', 'Mango', 'Papaya']
    
    # Image parameters
    IMG_SIZE = 224
    IMG_MEAN = [0.485, 0.456, 0.406]
    IMG_STD = [0.229, 0.224, 0.225]
    
    # Training parameters
    BATCH_SIZE = 24
    NUM_EPOCHS = 100
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-4
    
    # Augmentation parameters
    AUG_PROB = 0.5
    
    # Early stopping
    PATIENCE = 15
    MIN_DELTA = 1e-4
    
    # Data split
    TRAIN_RATIO = 0.7
    VAL_RATIO = 0.15
    TEST_RATIO = 0.15
    
    # Model architecture
    DROPOUT_RATE = 0.3
    
    # Logging
    LOG_INTERVAL = 10
    
    @classmethod
    def create_dirs(cls):
        """Create output directories."""
        cls.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        cls.CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
        cls.LOG_DIR.mkdir(parents=True, exist_ok=True)
        print("✅ Output directories created")

# Create directories
Config.create_dirs()

# Display configuration
print("\n📋 Configuration (DenseNet + Augmentation):")
print(f"   - Image Size: {Config.IMG_SIZE}x{Config.IMG_SIZE}")
print(f"   - Batch Size: {Config.BATCH_SIZE}")
print(f"   - Learning Rate: {Config.LEARNING_RATE}")
print(f"   - Epochs: {Config.NUM_EPOCHS}")
print(f"   - Augmentation: ENABLED (prob={Config.AUG_PROB})")
print(f"   - No FPN, No GCA, No Class Weighting")

In [ ]:
def load_dataset(data_root: Path) -> pd.DataFrame:
    """Load dataset from directory structure."""
    data = []
    
    for plant in Config.PLANTS:
        for class_name in Config.CLASSES:
            folder = data_root / plant / class_name
            if not folder.exists():
                print(f"⚠️  Folder not found: {folder}")
                continue
            
            for img_path in folder.glob('*.jpg'):
                data.append({
                    'path': str(img_path),
                    'plant': plant,
                    'class': class_name,
                    'label': Config.CLASSES.index(class_name)
                })
    
    df = pd.DataFrame(data)
    print(f"✅ Loaded {len(df)} images from {len(Config.PLANTS) * len(Config.CLASSES)} folders")
    
    # Display statistics
    print("\n📊 Dataset Statistics:")
    print(df.pivot_table(index='plant', columns='class', values='path', aggfunc='count'))
    print("\n📈 Overall Class Distribution:")
    print(df['class'].value_counts())
    
    return df

# Load dataset
df = load_dataset(Config.DATA_ROOT)

In [ ]:
# Split dataset into train/val/test
train_df, temp_df = train_test_split(
    df, 
    test_size=(1 - Config.TRAIN_RATIO),
    stratify=df['label'],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=Config.TEST_RATIO / (Config.VAL_RATIO + Config.TEST_RATIO),
    stratify=temp_df['label'],
    random_state=42
)

print(f"✅ Data split completed:")
print(f"   Training:   {len(train_df)} samples ({len(train_df)/len(df)*100:.1f}%)")
print(f"   Validation: {len(val_df)} samples ({len(val_df)/len(df)*100:.1f}%)")
print(f"   Test:       {len(test_df)} samples ({len(test_df)/len(df)*100:.1f}%)")

In [ ]:
# AUGMENTATION ENABLED
# Using Albumentations for advanced augmentation

train_transforms = A.Compose([
    A.Resize(Config.IMG_SIZE, Config.IMG_SIZE),
    
    # Geometric augmentations
    A.HorizontalFlip(p=Config.AUG_PROB),
    A.VerticalFlip(p=Config.AUG_PROB),
    A.Rotate(limit=45, p=Config.AUG_PROB),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=45, p=Config.AUG_PROB),
    
    # Color augmentations
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=Config.AUG_PROB),
    A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=Config.AUG_PROB),
    A.RGBShift(r_shift_limit=15, g_shift_limit=15, b_shift_limit=15, p=Config.AUG_PROB),
    
    # Noise and blur
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
    A.GaussianBlur(blur_limit=(3, 7), p=0.3),
    
    # Normalize and convert to tensor
    A.Normalize(mean=Config.IMG_MEAN, std=Config.IMG_STD),
    ToTensorV2()
])

val_transforms = A.Compose([
    A.Resize(Config.IMG_SIZE, Config.IMG_SIZE),
    A.Normalize(mean=Config.IMG_MEAN, std=Config.IMG_STD),
    ToTensorV2()
])

print("✅ Augmentation transforms created")
print("   Training augmentations:")
print("   - Horizontal/Vertical flip")
print("   - Rotation (±45°)")
print("   - Shift/Scale/Rotate")
print("   - Brightness/Contrast adjustment")
print("   - Hue/Saturation/Value changes")
print("   - RGB shifts")
print("   - Gaussian noise and blur")

In [ ]:
class AugmentedAnthracnoseDataset(Dataset):
    """Dataset with Albumentations augmentation."""
    
    def __init__(self, dataframe: pd.DataFrame, transforms=None):
        self.data = dataframe.reset_index(drop=True)
        self.transforms = transforms
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        
        # Load image
        image = Image.open(row['path']).convert('RGB')
        image = np.array(image)
        
        # Apply Albumentations transforms
        if self.transforms:
            augmented = self.transforms(image=image)
            image = augmented['image']
        
        label = row['label']
        
        return {
            'image': image,
            'label': label,
            'path': row['path'],
            'plant': row['plant']
        }

# Create datasets
train_dataset = AugmentedAnthracnoseDataset(train_df, transforms=train_transforms)
val_dataset = AugmentedAnthracnoseDataset(val_df, transforms=val_transforms)
test_dataset = AugmentedAnthracnoseDataset(test_df, transforms=val_transforms)

print(f"✅ Datasets created with augmentation:")
print(f"   Training:   {len(train_dataset)} samples (augmented)")
print(f"   Validation: {len(val_dataset)} samples (no augmentation)")
print(f"   Test:       {len(test_dataset)} samples (no augmentation)")

In [ ]:
# Create simple data loaders WITHOUT weighted sampling
train_loader = DataLoader(
    train_dataset,
    batch_size=Config.BATCH_SIZE,
    shuffle=True,  # Simple shuffle, no weighted sampling
    num_workers=0,
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=Config.BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=Config.BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print(f"✅ Data loaders created:")
print(f"   Training batches:   {len(train_loader)}")
print(f"   Validation batches: {len(val_loader)}")
print(f"   Test batches:       {len(test_loader)}")

# Verify a batch
sample_batch = next(iter(train_loader))
print(f"\n📦 Sample batch:")
print(f"   Image shape: {sample_batch['image'].shape}")
print(f"   Labels: {sample_batch['label'].tolist()}")

In [ ]:
class DenseNetWithAugmentation(nn.Module):
    """DenseNet201 classifier - NO FPN, NO GCA."""
    
    def __init__(self, num_classes: int = 2, pretrained: bool = True, dropout_rate: float = 0.3):
        super().__init__()
        
        # Load pretrained DenseNet201
        if pretrained:
            weights = DenseNet201_Weights.IMAGENET1K_V1
            self.backbone = densenet201(weights=weights)
        else:
            self.backbone = densenet201(weights=None)
        
        # Get number of features from backbone
        num_features = self.backbone.classifier.in_features
        
        # Replace classifier with simple head
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(num_features, num_classes)
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone(x)

# Create model
model = DenseNetWithAugmentation(
    num_classes=Config.NUM_CLASSES,
    pretrained=True,
    dropout_rate=Config.DROPOUT_RATE
).to(DEVICE)

# Print model summary
print("✅ Model created (DenseNet + Augmentation)")
print(f"\n📊 Model Summary:")
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"   Total parameters:     {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")
print(f"   Model size:           {total_params * 4 / (1024**2):.2f} MB")

# Test forward pass
with torch.no_grad():
    dummy_input = torch.randn(2, 3, Config.IMG_SIZE, Config.IMG_SIZE).to(DEVICE)
    dummy_output = model(dummy_input)
    print(f"   Input shape:  {dummy_input.shape}")
    print(f"   Output shape: {dummy_output.shape}")

In [ ]:
# SIMPLE CROSS ENTROPY LOSS - NO CLASS WEIGHTING, NO LABEL SMOOTHING
criterion = nn.CrossEntropyLoss()

print("✅ Simple CrossEntropyLoss (no class weights, no label smoothing)")

In [ ]:
# Optimizer
optimizer = Adam(
    model.parameters(),
    lr=Config.LEARNING_RATE,
    weight_decay=Config.WEIGHT_DECAY
)

# Simple scheduler
scheduler = ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=5,
    verbose=True
)

print("✅ Optimizer and scheduler created")

In [ ]:
class MetricsTracker:
    """Track training metrics."""
    
    def __init__(self):
        self.history = {
            'train_loss': [],
            'train_acc': [],
            'train_f1': [],
            'val_loss': [],
            'val_acc': [],
            'val_f1': [],
            'val_precision': [],
            'val_recall': [],
            'val_auc': [],
            'learning_rate': []
        }
        self.best_f1 = 0.0
        self.best_epoch = 0
    
    def update(self, metrics: Dict, epoch: int):
        for key, value in metrics.items():
            if key in self.history:
                self.history[key].append(value)
        
        if 'val_f1' in metrics and metrics['val_f1'] > self.best_f1:
            self.best_f1 = metrics['val_f1']
            self.best_epoch = epoch
    
    def save(self, path: Path):
        with open(path, 'w') as f:
            json.dump({
                'history': self.history,
                'best_f1': self.best_f1,
                'best_epoch': self.best_epoch
            }, f, indent=2)

print("✅ MetricsTracker defined")

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    
    pbar = tqdm(dataloader, desc="Training")
    for batch in pbar:
        images = batch['image'].to(device)
        labels = batch['label'].to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Track metrics
        total_loss += loss.item()
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
        
        pbar.set_postfix({'loss': loss.item()})
    
    # Calculate metrics
    avg_loss = total_loss / len(dataloader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    
    return {
        'train_loss': avg_loss,
        'train_acc': acc,
        'train_f1': f1
    }

print("✅ train_epoch function defined")

In [ ]:
@torch.no_grad()
def validate_epoch(model, dataloader, criterion, device):
    """Validate for one epoch."""
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    all_probs = []
    
    pbar = tqdm(dataloader, desc="Validation")
    for batch in pbar:
        images = batch['image'].to(device)
        labels = batch['label'].to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Track metrics
        total_loss += loss.item()
        probs = F.softmax(outputs, dim=1).cpu().numpy()
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_probs.extend(probs)
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
        
        pbar.set_postfix({'loss': loss.item()})
    
    # Calculate metrics
    avg_loss = total_loss / len(dataloader)
    acc = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    
    # AUC
    all_probs = np.array(all_probs)
    try:
        auc = roc_auc_score(all_labels, all_probs[:, 1])
    except:
        auc = 0.0
    
    return {
        'val_loss': avg_loss,
        'val_acc': acc,
        'val_precision': precision,
        'val_recall': recall,
        'val_f1': f1,
        'val_auc': auc
    }

print("✅ validate_epoch function defined")

In [ ]:
def train_model(model, train_loader, val_loader, config, device):
    """Complete training loop."""
    metrics_tracker = MetricsTracker()
    best_model_state = None
    patience_counter = 0
    
    print("\n" + "="*60)
    print("🚀 STARTING TRAINING (DenseNet + Augmentation)")
    print("="*60)
    
    for epoch in range(1, config.NUM_EPOCHS + 1):
        print(f"\n{'─'*60}")
        print(f"📅 Epoch {epoch}/{config.NUM_EPOCHS}")
        print(f"   Learning Rate: {optimizer.param_groups[0]['lr']:.2e}")
        print(f"{'─'*60}")
        
        # Train
        train_metrics = train_epoch(model, train_loader, criterion, optimizer, device)
        
        # Validate
        val_metrics = validate_epoch(model, val_loader, criterion, device)
        
        # Update scheduler
        scheduler.step(val_metrics['val_loss'])
        
        # Combine metrics
        epoch_metrics = {**train_metrics, **val_metrics}
        epoch_metrics['learning_rate'] = optimizer.param_groups[0]['lr']
        metrics_tracker.update(epoch_metrics, epoch)
        
        # Print summary
        print(f"\n📊 Epoch {epoch} Summary:")
        print(f"   Train Loss: {train_metrics['train_loss']:.4f} | Train Acc: {train_metrics['train_acc']:.4f}")
        print(f"   Val Loss:   {val_metrics['val_loss']:.4f} | Val Acc:   {val_metrics['val_acc']:.4f}")
        print(f"   Val F1:     {val_metrics['val_f1']:.4f} | Val AUC:   {val_metrics['val_auc']:.4f}")
        
        # Save best model
        if val_metrics['val_f1'] > metrics_tracker.best_f1:
            best_model_state = model.state_dict().copy()
            torch.save({
                'epoch': epoch,
                'model_state_dict': best_model_state,
                'optimizer_state_dict': optimizer.state_dict(),
                'metrics': val_metrics
            }, config.CHECKPOINT_DIR / 'best_model.pth')
            print(f"   ✅ New best model saved! (F1: {val_metrics['val_f1']:.4f})")
            patience_counter = 0
        else:
            patience_counter += 1
            print(f"   ⏳ EarlyStopping: {patience_counter}/{config.PATIENCE}")
        
        # Early stopping
        if patience_counter >= config.PATIENCE:
            print(f"\n⛔ Early stopping at epoch {epoch}")
            break
    
    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"\n✅ Loaded best model with F1: {metrics_tracker.best_f1:.4f}")
    
    # Save training history
    metrics_tracker.save(config.LOG_DIR / 'training_history.json')
    print(f"✅ Training history saved to {config.LOG_DIR / 'training_history.json'}")
    
    print("\n" + "="*60)
    print("🏁 TRAINING COMPLETE")
    print("="*60)
    print(f"\n🏆 Best Results (Epoch {metrics_tracker.best_epoch}):")
    print(f"   Best Val F1 Score: {metrics_tracker.best_f1:.4f}")
    
    return model, metrics_tracker

print("✅ Training function defined")

In [ ]:
# RUN TRAINING
trained_model, metrics = train_model(model, train_loader, val_loader, Config, DEVICE)

In [ ]:
def plot_training_curves(metrics_tracker, save_path=None):
    """Plot training curves."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    epochs = range(1, len(metrics_tracker.history['train_loss']) + 1)
    
    # Loss
    axes[0, 0].plot(epochs, metrics_tracker.history['train_loss'], 'b-', label='Train', linewidth=2)
    axes[0, 0].plot(epochs, metrics_tracker.history['val_loss'], 'r-', label='Val', linewidth=2)
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Accuracy
    axes[0, 1].plot(epochs, metrics_tracker.history['train_acc'], 'b-', label='Train', linewidth=2)
    axes[0, 1].plot(epochs, metrics_tracker.history['val_acc'], 'r-', label='Val', linewidth=2)
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Accuracy')
    axes[0, 1].set_title('Accuracy')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # F1 Score
    axes[1, 0].plot(epochs, metrics_tracker.history['train_f1'], 'b-', label='Train', linewidth=2)
    axes[1, 0].plot(epochs, metrics_tracker.history['val_f1'], 'r-', label='Val', linewidth=2)
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('F1 Score')
    axes[1, 0].set_title('F1 Score')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Learning Rate
    axes[1, 1].plot(epochs, metrics_tracker.history['learning_rate'], 'g-', linewidth=2)
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Learning Rate')
    axes[1, 1].set_title('Learning Rate')
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].set_yscale('log')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"✅ Curves saved to {save_path}")
    
    plt.show()

# Plot curves
plot_training_curves(metrics, Config.OUTPUT_DIR / 'training_curves.png')

In [ ]:
# Save final model
final_checkpoint = {
    'model_state_dict': trained_model.state_dict(),
    'config': {
        'IMG_SIZE': Config.IMG_SIZE,
        'NUM_CLASSES': Config.NUM_CLASSES,
        'BATCH_SIZE': Config.BATCH_SIZE,
        'LEARNING_RATE': Config.LEARNING_RATE,
        'NUM_EPOCHS': Config.NUM_EPOCHS,
        'DROPOUT_RATE': Config.DROPOUT_RATE,
        'AUGMENTATION': True
    },
    'final_metrics': {
        'best_f1': metrics.best_f1,
        'best_epoch': metrics.best_epoch
    },
    'timestamp': datetime.now().isoformat()
}

torch.save(final_checkpoint, Config.OUTPUT_DIR / 'final_model.pth')
print(f"✅ Final model saved to {Config.OUTPUT_DIR / 'final_model.pth'}")

# Save config
config_dict = {
    'model': 'DenseNet201_with_Augmentation',
    'features': 'Augmentation ENABLED, No FPN, No Class Weighting, No GCA',
    'IMG_SIZE': Config.IMG_SIZE,
    'NUM_CLASSES': Config.NUM_CLASSES,
    'BATCH_SIZE': Config.BATCH_SIZE,
    'LEARNING_RATE': Config.LEARNING_RATE,
    'DROPOUT_RATE': Config.DROPOUT_RATE,
    'AUGMENTATION': True,
    'AUG_PROB': Config.AUG_PROB
}

with open(Config.OUTPUT_DIR / 'config.json', 'w') as f:
    json.dump(config_dict, f, indent=2)
print(f"✅ Configuration saved to {Config.OUTPUT_DIR / 'config.json'}")

print("\n" + "="*60)
print("✅ TRAINING COMPLETE (DenseNet + Augmentation)")
print("="*60)